In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CBGOC10", con=connection2)

connection2.close()




In [2]:
base2

,grupocupcod,grupocupnom,grupocupnomcor,grupocupotorreceflg,grupocupotorcittflg,grupocupotorexamflg,grupocupotormateflg,grupocupotorrefeflg,grupocupreganteflg,grupocupresimgflg,grupocupreslabflg,grupocuprespatflg,grupocupeqvssaludcod,grupocupeqvfacturcod,grupocuprevimgflg,grupocuprevlabflg,grupocuprevpatflg
0,01,MEDICO,MEDICO,1,1,1,1,1,1,1,1,1,01,1,1,1,1
1,02,ODONTOLOGO,ODONTOLOGO,1,1,1,1,1,1,0,0,0,03,3,None,None,None
2,03,OBSTETRIZ,OBSTETRIZ,1,1,1,1,1,1,0,0,0,05,5,None,None,None
3,04,PSICOLOGO,PSICOLOGO,1,0,0,0,0,0,0,0,0,08,8,None,None,None
4,05,NUTRICIONISTA,NUTRICIONISTA,1,0,0,0,0,0,0,0,0,10,10,None,None,None
5,06,TRABAJADOR(A) SOCIAL,TRAB.SOCIAL,0,0,0,0,0,0,0,0,0,07,7,None,None,None
6,07,QUIMICO FARMACEUTICO,QUIMICO FARM.,0,0,0,0,0,0,0,0,0,02,2,None,None,None
7,08,BIOLOGO,BIOLOGO,0,0,0,0,0,0,0,1,1,04,4,0,1,1
8,09,ENFERMERA(O),ENFERMERA(O),1,0,0,1,0,0,0,0,0,06,6,None,None,None
9,21,TECNOLOGO MEDICO EN TERAPIA FIS.Y REHAB.,TECN.MED.REHAB.,0,0,0,0,0,0,0,0,0,09,9,None,None,None


In [3]:
a=base2[base2.duplicated(subset=["grupocupcod"])]
a

,grupocupcod,grupocupnom,grupocupnomcor,grupocupotorreceflg,grupocupotorcittflg,grupocupotorexamflg,grupocupotormateflg,grupocupotorrefeflg,grupocupreganteflg,grupocupresimgflg,grupocupreslabflg,grupocuprespatflg,grupocupeqvssaludcod,grupocupeqvfacturcod,grupocuprevimgflg,grupocuprevlabflg,grupocuprevpatflg


In [4]:
base2.columns

Index(['grupocupcod', 'grupocupnom', 'grupocupnomcor', 'grupocupotorreceflg',
       'grupocupotorcittflg', 'grupocupotorexamflg', 'grupocupotormateflg',
       'grupocupotorrefeflg', 'grupocupreganteflg', 'grupocupresimgflg',
       'grupocupreslabflg', 'grupocuprespatflg', 'grupocupeqvssaludcod',
       'grupocupeqvfacturcod', 'grupocuprevimgflg', 'grupocuprevlabflg',
       'grupocuprevpatflg'],
      dtype='object')

In [5]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cbgoc10 ()')
base2.to_sql(name='tmp_cbgoc10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cbgoc10 FALSO CON LO OBTENIDO DEL ORACLE

query="""
ALTER TABLE tmp_cbgoc10 
ALTER COLUMN grupocupcod TYPE CHARACTER (2), 
ALTER COLUMN grupocupnom TYPE CHARACTER (40), 
ALTER COLUMN grupocupnomcor TYPE CHARACTER (15), 
ALTER COLUMN grupocupotorreceflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupotorcittflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupotorexamflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupotormateflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupotorrefeflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupreganteflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupresimgflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupreslabflg TYPE CHARACTER (1), 
ALTER COLUMN grupocuprespatflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupeqvssaludcod TYPE CHARACTER (2), 
ALTER COLUMN grupocupeqvfacturcod TYPE CHARACTER (2), 
ALTER COLUMN grupocuprevimgflg TYPE CHARACTER (1), 
ALTER COLUMN grupocuprevlabflg TYPE CHARACTER (1), 
ALTER COLUMN grupocuprevpatflg TYPE CHARACTER (1); 


UPDATE cbgoc10 
SET 


grupocupcod             =t.grupocupcod ,
grupocupnom             =t.grupocupnom ,
grupocupnomcor          =t.grupocupnomcor ,
grupocupotorreceflg             =t.grupocupotorreceflg ,
grupocupotorcittflg             =t.grupocupotorcittflg ,
grupocupotorexamflg             =t.grupocupotorexamflg ,
grupocupotormateflg             =t.grupocupotormateflg ,
grupocupotorrefeflg             =t.grupocupotorrefeflg ,
grupocupreganteflg          =t.grupocupreganteflg ,
grupocupresimgflg           =t.grupocupresimgflg ,
grupocupreslabflg           =t.grupocupreslabflg ,
grupocuprespatflg           =t.grupocuprespatflg ,
grupocupeqvssaludcod            =t.grupocupeqvssaludcod ,
grupocupeqvfacturcod            =t.grupocupeqvfacturcod ,
grupocuprevimgflg           =t.grupocuprevimgflg ,
grupocuprevlabflg           =t.grupocuprevlabflg ,
grupocuprevpatflg           =t.grupocuprevpatflg 



FROM tmp_cbgoc10 t 
WHERE cbgoc10.grupocupcod = t.grupocupcod and cbgoc10.grupocupcod IS NOT NULL and t.grupocupcod IS NOT NULL ;

INSERT INTO cbgoc10 
(grupocupcod, grupocupnom, grupocupnomcor, grupocupotorreceflg,
grupocupotorcittflg, grupocupotorexamflg, grupocupotormateflg,
grupocupotorrefeflg, grupocupreganteflg, grupocupresimgflg,
grupocupreslabflg, grupocuprespatflg, grupocupeqvssaludcod,
grupocupeqvfacturcod, grupocuprevimgflg, grupocuprevlabflg,
grupocuprevpatflg) 
SELECT 
grupocupcod, grupocupnom, grupocupnomcor, grupocupotorreceflg,
grupocupotorcittflg, grupocupotorexamflg, grupocupotormateflg,
grupocupotorrefeflg, grupocupreganteflg, grupocupresimgflg,
grupocupreslabflg, grupocuprespatflg, grupocupeqvssaludcod,
grupocupeqvfacturcod, grupocuprevimgflg, grupocuprevlabflg,
grupocuprevpatflg

FROM tmp_cbgoc10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cbgoc10 
    WHERE cbgoc10.grupocupcod = tmp_cbgoc10.grupocupcod and cbgoc10.grupocupcod IS NOT NULL and tmp_cbgoc10.grupocupcod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cbgoc10;
DELETE FROM cbgoc10 WHERE grupocupcod ='';
"""


c2= text(query2)
connection3.execute(c2)
base2 = pd.read_sql_query(f"SELECT * FROM cbgoc10", con=connection3)

connection3.close()


In [6]:
base2.columns

Index(['grupocupcod', 'grupocupnom', 'grupocupnomcor', 'grupocupotorreceflg',
       'grupocupotorcittflg', 'grupocupotorexamflg', 'grupocupotormateflg',
       'grupocupotorrefeflg', 'grupocupreganteflg', 'grupocupresimgflg',
       'grupocupreslabflg', 'grupocuprespatflg', 'grupocupeqvssaludcod',
       'grupocupeqvfacturcod', 'grupocuprevimgflg', 'grupocuprevlabflg',
       'grupocuprevpatflg'],
      dtype='object')

In [7]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cbgoc10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query="""

ALTER TABLE tmp_cbgoc10 
ALTER COLUMN grupocupcod TYPE CHARACTER (2), 
ALTER COLUMN grupocupnom TYPE CHARACTER (40), 
ALTER COLUMN grupocupnomcor TYPE CHARACTER (15), 
ALTER COLUMN grupocupotorreceflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupotorcittflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupotorexamflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupotormateflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupotorrefeflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupreganteflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupresimgflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupreslabflg TYPE CHARACTER (1), 
ALTER COLUMN grupocuprespatflg TYPE CHARACTER (1), 
ALTER COLUMN grupocupeqvssaludcod TYPE CHARACTER (2), 
ALTER COLUMN grupocupeqvfacturcod TYPE CHARACTER (2), 
ALTER COLUMN grupocuprevimgflg TYPE CHARACTER (1), 
ALTER COLUMN grupocuprevlabflg TYPE CHARACTER (1), 
ALTER COLUMN grupocuprevpatflg TYPE CHARACTER (1); 


UPDATE dim_gruocu 
SET 

des_gru       = t.grupocupnom,
dco_gru       = t.grupocupnomcor

FROM tmp_cbgoc10 t 
WHERE dim_gruocu.cod_gru = t.grupocupcod AND dim_gruocu.cod_gru  IS NOT NULL;


INSERT INTO dim_gruocu (cod_gru, des_gru, dco_gru) 
SELECT grupocupcod, grupocupnom, grupocupnomcor      

FROM tmp_cbgoc10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_gruocu 
    WHERE dim_gruocu.cod_gru = tmp_cbgoc10.grupocupcod
);

DROP TABLE tmp_cbgoc10;

"""



c1= text(query)
connection4.execute(c1)
connection4.close()